# 03 · Directed propagation graph

The novel artefact is an 8-neighbour spatial lattice with edge travel-time costs, not a learned GNN. Wind and slope create asymmetric paths.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import json
import matplotlib.pyplot as plt
INPUT = ROOT / "data/synthetic/toy_firms.csv"  # SYNTHETIC DEMO, NOT historical NASA data
WEATHER = ROOT / "data/synthetic/toy_weather.json"
assert INPUT.exists(), "Run `python scripts/make_synthetic.py` from repository root first"
print("Data mode: SYNTHETIC - demonstration only")

In [ ]:
from wildfire.ingest import load_firms, load_weather,parse_utc
from wildfire.features import cluster_detections,weather_summary
from wildfire.graph import build_grid,build_adjacency,SpreadParameters,reverse_shortest_paths
from datetime import timedelta
clusters=cluster_detections(load_firms(INPUT))
grid=build_grid(clusters,cell_m=850,margin_km=9)
first=parse_utc(clusters[0]['timestamp_utc'])
end=(first+timedelta(hours=10)).isoformat().replace('+00:00','Z')
wind=weather_summary(load_weather(WEATHER),first.isoformat().replace('+00:00','Z'),end)
adj,stats=build_adjacency(grid,wind,SpreadParameters())
print('Graph stats:',stats)
print('Projected grid range:',grid.nx,'x',grid.ny)

In [ ]:
# Show a small subset of physically weighted arrow edges.
fig,ax=plt.subplots(figsize=(10,6))
step=max(1,len(grid.nodes)//280)
for n in grid.nodes[::step]:
    ax.scatter(n.lon,n.lat,s=3,c='lightgray')
for c in clusters:
    ax.scatter(c['longitude'],c['latitude'],s=80,marker='x',c='darkred')
ax.set(xlabel='Longitude',ylabel='Latitude',title='Candidate lattice and observed fire clusters')
plt.show()

In [ ]:
# Check that wind creates directional edges: eastward spread faster than westward.
node=grid.nearest(clusters[0]['latitude'],clusters[0]['longitude'])
next_east=node+1
forward=dict(adj[node]).get(next_east)
backward=dict(adj[next_east]).get(node)
print('Travel east (hours):',forward,'west:',backward,'eastward faster:',forward<backward)

**Model limitation:** The exponential spread-speed law is a sensitivity parameterisation, not a fully calibrated wildfire simulation. Future work: dynamic wind, validated rate-of-spread laws and pre-fire fuel mapping.